In [1]:
import random
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# CONFIGURATION
# =========================
NUM_SAMPLES = 665

# Output paths (single dataset, no splits)
OUT_DIR = Path("../../../data/simulation")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_OUTPUT_PATH = OUT_DIR / "engine_start.csv"
X_OUTPUT_PATH   = OUT_DIR / "engine_start_X.npy"
Y_OUTPUT_PATH   = OUT_DIR / "engine_start_y.npy"

FEATURE_COLS = ["Temperature", "Pressure", "RPM", "Vibration"]
WINDOW_LEN = 4

# =========================
# CHANGES vs training generator (SIMULATION REALISM ONLY):
#   1. cold/cooling is picked ONCE PER WINDOW, not per timestep --
#      no more 27C -> 105C jumps inside one window. OFF temps evolve
#      smoothly (cold: random walk +-0.07/s; cooling: slow descent),
#      matching the engine_off generator's physics.
#   2. Temperature during the ON ramp is capped at 110C -- a start
#      never climbs into CriticalLoad range (>=115C) within its own
#      window. Hot restarts (cooling ~110C) stay hot but don't heat up.
# The TRAINING generator is intentionally left untouched (broader
# envelope = robustness); this file only shapes the benchmark scenario.
# =========================

TEMP_CAP_ON = 110.0  # below CriticalLoad range (115-145)

def gen_off_start_conditions():
    """
    Pick the OFF regime ONCE for the whole window and return
    (state, initial_temperature). Ranges match the training generator:
    cold [-10, 35], cooling [60, 120].
    """
    state = random.choice([0, 1])  # 0=cold, 1=cooling
    if state == 0:
        temperature = random.uniform(-10, 35)
    else:
        temperature = random.uniform(60, 120)
    return state, temperature


def gen_off_timestep(state, temperature):
    """
    One OFF timestep for a GIVEN regime and current temperature.
    - cold:    T random-walks +-0.07/s (same as engine_off generator)
    - cooling: T descends slowly, -0.5..-0.1/s (same as engine_off generator)
    - Pressure: uniform [0.92, 1.02]
    - RPM: 0.0
    - Vibration: normal low; if temp >= 95 (cooling), slightly higher
    Returns (row, next_temperature).
    """
    if state == 0:
        temperature += random.uniform(-0.07, 0.07)
        vibration = random.uniform(0.0001, 0.001)
    else:
        temperature += random.uniform(-0.5, -0.1)
        if temperature >= 95:
            vibration = random.uniform(0.002, 0.005)
        else:
            vibration = random.uniform(0.0001, 0.001)

    pressure = random.uniform(0.92, 1.02)
    rpm = 0.0
    return [temperature, pressure, rpm, vibration], temperature

# =========================
# GENERATE ONE ENGINE START WINDOW (4 timesteps)
# =========================
def gen_engine_start_window():
    """
    Build a 4-step window:
      - n_off in {1,2,3} OFF timesteps first, ALL from the same regime
        (cold or cooling), evolving smoothly.
      - Remaining steps are ON with:
          * Temp += 3..7 each ON step, CAPPED at 110C
          * RPM first ON: 1500..3000
          * RPM second ON: previous - (400..600)
          * RPM third ON (if exists): previous - (100..300)
          * Pressure on first ON: 0.7..0.9 (drop), then normal 0.92..1.02
          * Vibration on first ON: 0.3..0.6 (spike), then 0.05..0.15 (settle)
    Label: "Engine Start"
    """
    n_off = random.choice([1, 2, 3])
    n_on = WINDOW_LEN - n_off

    window = []

    # OFF part -- one regime for the whole window
    state, temp0 = gen_off_start_conditions()
    last_temp = temp0
    for _ in range(n_off):
        row, last_temp = gen_off_timestep(state, last_temp)
        window.append(row)

    # ON part
    if n_on > 0:
        # First ON step
        temp = min(last_temp + random.uniform(3, 7), TEMP_CAP_ON)
        rpm = random.uniform(1500, 3000)
        pressure = random.uniform(0.7, 0.9)       # pressure drop
        vibration = random.uniform(0.3, 0.6)      # vibration spike
        window.append([temp, pressure, rpm, vibration])

        # Second ON step (if exists): drop 400..600
        if n_on >= 2:
            temp = min(temp + random.uniform(3, 7), TEMP_CAP_ON)
            rpm = max(0.0, rpm - random.uniform(400, 600))
            pressure = random.uniform(0.9, 0.95)  # back to normal range
            vibration = random.uniform(0.15, 0.3) # settle
            window.append([temp, pressure, rpm, vibration])

        # Third ON step (if exists): drop 100..300
        if n_on == 3:
            temp = min(temp + random.uniform(3, 7), TEMP_CAP_ON)
            rpm = max(0.0, rpm - random.uniform(100, 300))
            pressure = random.uniform(0.98, 1.02)  # back to normal range
            vibration = random.uniform(0.05, 0.15) # settle
            window.append([temp, pressure, rpm, vibration])

    assert len(window) == WINDOW_LEN
    return window, "Engine Start"

# =========================
# GENERATE DATASET
# =========================
def generate_dataset(num_samples):
    X_list = []
    y_list = []
    rows = []

    for seq_num in range(num_samples):
        window, label = gen_engine_start_window()
        X_list.append(window)
        y_list.append(label)

        # For CSV: long format with Time, Sequence, variables, State
        for t_idx, (temp, pres, rpm, vib) in enumerate(window, start=1):
            rows.append({
                "Time": t_idx,               # 1..WINDOW_LEN
                "Sequence": seq_num + 1,     # 1..num_samples
                "Temperature": temp,
                "Pressure": pres,
                "RPM": rpm,
                "Vibration": vib,
                "State": label               # keep label as-is: "Engine Start"
            })

    X = np.array(X_list, dtype=np.float32)  # (N, 4, 4)
    y = np.array(y_list)                    # (N,)
    df = pd.DataFrame(rows, columns=["Time","Sequence","Temperature","Pressure","RPM","Vibration","State"])
    return X, y, df

# =========================
# SAVE (single dataset)
# =========================
X, y, df = generate_dataset(NUM_SAMPLES)
np.save(X_OUTPUT_PATH, X)
np.save(Y_OUTPUT_PATH, y)
df.to_csv(CSV_OUTPUT_PATH, index=False)

print("✅ Engine Start dataset created (simulation, physically coherent).")
print("X:", X.shape, "| y:", y.shape)

✅ Engine Start dataset created (simulation, physically coherent).
X: (665, 4, 4) | y: (665,)
